# BHaH Project Anatomy

BHaH is NRPy's standalone generated-project backend. Earlier wave-equation notebooks use it to produce a C project that can be built with ordinary compiler and `make` tooling. This notebook inspects that project layout without repeating the full build-and-run workflow.

## Table of Contents

1. [Generator discovery](#Generator-discovery)
2. [Generate a fresh BHaH project](#Generate-a-fresh-BHaH-project)
3. [Confirm core generated files](#Confirm-core-generated-files)
4. [Map files to responsibilities](#Map-files-to-responsibilities)
5. [Inspect structs, parameters, and entry points](#Inspect-structs,-parameters,-and-entry-points)
6. [Trace registered functions into C](#Trace-registered-functions-into-C)

## Generator Discovery

The Cartesian wave-equation generator is a project entry point. Discovery uses `importlib.util.find_spec` so the module is located without importing it at notebook startup.

In [ ]:
import importlib.util
import re
import subprocess
import sys
import tempfile
from pathlib import Path

import nrpy

GENERATOR_MODULE = "nrpy.examples.wave_equation_cartesian"
PROJECT_NAME = "wave_equation_cartesian"

print("nrpy package:", nrpy.__file__)
generator_spec = importlib.util.find_spec(GENERATOR_MODULE)
print("generator discovered:", generator_spec is not None)
if generator_spec is None:
    raise RuntimeError(f"cannot locate {GENERATOR_MODULE}")


def compact_output(text, max_lines=12):
    text = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", text)
    lines = [line.rstrip() for line in text.replace("\r", "\n").splitlines() if line.strip()]
    if len(lines) > max_lines:
        lines = ["..."] + lines[-max_lines:]
    return "\n".join(lines)

## Generate a Fresh BHaH Project

The generator writes `project/wave_equation_cartesian/` relative to its working directory. A temporary workspace keeps existing projects untouched.

In [ ]:
workspace_manager = tempfile.TemporaryDirectory(prefix="nrpy-bhah-")
WORKSPACE = Path(workspace_manager.name).resolve()
PROJECT_DIR = WORKSPACE / "project" / PROJECT_NAME

command = [sys.executable, "-m", GENERATOR_MODULE]
result = subprocess.run(
    command,
    cwd=WORKSPACE,
    text=True,
    capture_output=True,
    timeout=300,
)

print("command:", " ".join(command))
print("return code:", result.returncode)
if result.stdout:
    print("stdout:")
    print(compact_output(result.stdout))
if result.stderr:
    print("stderr:")
    print(compact_output(result.stderr))
if result.returncode != 0:
    raise RuntimeError("BHaH project generation failed")

print("project directory:", PROJECT_DIR)

## Confirm Core Generated Files

A minimal BHaH inspection starts with the build recipe, central header, C prototypes, and default parameter file.

In [ ]:
required_files = [
    "Makefile",
    "BHaH_defines.h",
    "BHaH_function_prototypes.h",
    f"{PROJECT_NAME}.par",
]

missing = []
for relative_path in required_files:
    path = PROJECT_DIR / relative_path
    exists = path.exists()
    print(f"{relative_path}: {exists}")
    if not exists:
        missing.append(relative_path)

if missing:
    raise RuntimeError("missing generated files: " + ", ".join(missing))

## Map Files to Responsibilities

The generated tree separates build logic, runtime parameters, grid/commondata structs, C-function implementations, diagnostics, and the executable entry point.

In [ ]:
responsibilities = {
    "Makefile": "builds the executable from generated C sources",
    "BHaH_defines.h": "collects shared definitions, macros, structs, and gridfunction metadata",
    "BHaH_function_prototypes.h": "declares generated C functions",
    f"{PROJECT_NAME}.par": "stores default runtime parameters",
    "main.c": "defines the executable entry point",
    "griddata_free.c": "releases generated grid/commondata allocations",
    "numerical_grids_and_timestep.c": "sets grid spacing, coordinates, and the time step",
    "diagnostics.c": "writes compact runtime diagnostics",
    "cmdline_input_and_parfile_parser.c": "loads parameter files and command-line overrides",
}

for relative_path, role in responsibilities.items():
    path = PROJECT_DIR / relative_path
    line_count = len(path.read_text(encoding="utf-8").splitlines()) if path.exists() else 0
    print(f"{relative_path}: {line_count} lines")
    print(f"  {role}")

## Inspect Structs, Parameters, and Entry Points

The central header carries generated structs; the parameter file carries runtime defaults; `main.c` and `diagnostics.c` show the executable and diagnostic entry points.

In [ ]:
def print_block(path, start_text, end_text, max_lines=24):
    lines = path.read_text(encoding="utf-8").splitlines()
    start = next((idx for idx, line in enumerate(lines) if start_text in line), None)
    if start is None:
        raise RuntimeError(f"cannot find {start_text} in {path.name}")
    end = next(
        (idx for idx in range(start, min(len(lines), start + max_lines)) if end_text in lines[idx]),
        min(len(lines), start + max_lines) - 1,
    )
    print(f"\n{path.name}: {start_text}")
    for line in lines[start : end + 1]:
        print(line)


defines_path = PROJECT_DIR / "BHaH_defines.h"
print_block(defines_path, "typedef struct __commondata_struct__", "} commondata_struct;")
print_block(defines_path, "typedef struct __params_struct__", "} params_struct;")
print_block(defines_path, "typedef struct __griddata__", "} griddata_struct;")

parameter_lines = [
    line
    for line in (PROJECT_DIR / f"{PROJECT_NAME}.par").read_text(encoding="utf-8").splitlines()
    if line.strip() and not line.lstrip().startswith("#")
]
print("\nparameter entries:")
for line in parameter_lines:
    print(line)

for relative_path, start_text in [
    ("main.c", "int main("),
    ("diagnostics.c", "void diagnostics("),
]:
    source_path = PROJECT_DIR / relative_path
    source_lines = source_path.read_text(encoding="utf-8").splitlines()
    start = next((idx for idx, line in enumerate(source_lines) if start_text in line), None)
    if start is None:
        raise RuntimeError(f"cannot find {start_text} in {relative_path}")
    print(f"\n{relative_path}: {start_text}")
    for line in source_lines[start : start + 12]:
        print(line)

## Trace Registered Functions Into C

NRPy's Python generator registers C functions; BHaH then emits source files, prototypes, and Makefile object entries. The representative functions below show that path.

In [ ]:
prototype_text = (PROJECT_DIR / "BHaH_function_prototypes.h").read_text(encoding="utf-8")
makefile_text = (PROJECT_DIR / "Makefile").read_text(encoding="utf-8")
generator_source_text = Path(generator_spec.origin).read_text(encoding="utf-8")

registered_functions = {
    "initial_data": "register_CFunction_initial_data",
    "rhs_eval": "register_CFunction_rhs_eval",
    "diagnostics": "register_CFunction_diagnostics",
    "numerical_grids_and_timestep": "register_CFunction_numerical_grids_and_timestep_setup",
    "cmdline_input_and_parfile_parser": "register_CFunction_cmdline_input_and_parfile_parser",
    "main": "register_CFunction_main_c",
}

missing_trace = []
for function_name, registration_name in registered_functions.items():
    source_file = PROJECT_DIR / f"{function_name}.c"
    object_name = f"{function_name}.o"
    print(f"\n{function_name}")
    print("  generator registration present:", registration_name in generator_source_text)
    print("  prototype present:", function_name in prototype_text)
    print("  source file present:", source_file.exists())
    print("  Makefile object present:", object_name in makefile_text)
    if not (
        registration_name in generator_source_text
        and function_name in prototype_text
        and source_file.exists()
        and object_name in makefile_text
    ):
        missing_trace.append(function_name)

if missing_trace:
    raise RuntimeError("incomplete registration trace: " + ", ".join(missing_trace))

print("\nMoL sources:")
for path in sorted((PROJECT_DIR / "MoL").glob("*.c")):
    print(" ", path.relative_to(PROJECT_DIR))

The BHaH anatomy is now visible: Python registration produces C functions and prototypes, the Makefile builds them into the executable, the parameter file controls runtime defaults, and the generated headers connect gridfunction and parameter metadata to every source file. See the earlier Cartesian wave project notebook for the full compile-and-run workflow.